In [1]:
import numpy as np

left_camera_id = 0
right_camera_id = 1

# Load the camera calibration
data = np.load('../calibration/logitech_brio_100.npz')
camera_matrix = data['mtx']
dist_coeffs = data['dist']

R = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])  # Rotation matrix
T = np.array([0.085, 0, 0])  # Translation vector

width = 1920
height = 1080

In [3]:
import cv2

rectify_scale = 1  # Should be 0 if size is the same, 1 if not
R1, R2, P1, P2, _, _, _ = cv2.stereoRectify(
    camera_matrix, dist_coeffs,
    camera_matrix, dist_coeffs,
    (width, height), R, T,
    rectify_scale, (0, 0)
)

left_map1, left_map2 = cv2.initUndistortRectifyMap(
    camera_matrix, dist_coeffs, R1, P1, (width, height), 5
)
right_map1, right_map2 = cv2.initUndistortRectifyMap(
    camera_matrix, dist_coeffs, R2, P2, (width, height), 5
)

In [5]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

left_camera = cv2.VideoCapture(left_camera_id)
right_camera = cv2.VideoCapture(right_camera_id)
left_camera.set(cv2.CAP_PROP_FRAME_WIDTH, width)
left_camera.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
right_camera.set(cv2.CAP_PROP_FRAME_WIDTH, width)
right_camera.set(cv2.CAP_PROP_FRAME_HEIGHT, height)


prev_gray_left = None
while True:
    # Capture frames from both cameras
    ret_left, frame_left = left_camera.read()
    ret_right, frame_right = right_camera.read()

    if not ret_left or not ret_right:
        break
    
    # Rectify the images
    rectified_left = cv2.remap(frame_left, left_map1, left_map2, cv2.INTER_LINEAR)
    rectified_right = cv2.remap(frame_right, right_map1, right_map2, cv2.INTER_LINEAR)

    # Convert frames to grayscale
    gray_left = cv2.cvtColor(rectified_left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(rectified_right, cv2.COLOR_BGR2GRAY)
    gray_left = np.uint8(gray_left)
    gray_right = np.uint8(gray_right)

    # Calculate disparity map and depth map
    stereo = cv2.StereoBM.create(numDisparities=16, blockSize=15)
    disparity = stereo.compute(gray_left, gray_right).astype(np.float32)
    valid_mask = disparity > 0
    depth_map = np.zeros_like(disparity)
    focal_length = camera_matrix[0, 0]
    baseline = T[0]
    depth_map[valid_mask] = (focal_length * baseline) / disparity[valid_mask]

    plt.figure(figsize=(10, 10))
    clear_output(wait=True)

    # Display depth map and left camera 
    plt.subplot(1, 2, 1)
    plt.axis('off') 
    plt.title('Depth Map')
    plt.imshow(depth_map, cmap='hot')

    plt.subplot(1, 2, 2)
    plt.axis('off') 
    plt.title('Right Camera')
    plt.imshow(cv2.cvtColor(rectified_right, cv2.COLOR_BGR2RGB))

    plt.show()



KeyboardInterrupt: 

In [20]:
left_camera.release()
right_camera.release() 
